In [1]:
import torch as trc
from torch.utils.data import Dataset, DataLoader
import os
import pandas as pd
import numpy as np
import random

In [2]:
device = trc.device("cuda" if trc.cuda.is_available() else "cpu")
print(device)

cuda


In [4]:
import datasetLoader

allfiles = [f[4:] for f in os.listdir(r"D:\Resume Projs\All-Weather-Land-Cover-Intelligence-using-SAR-Optical-Fusion\data\dataset\optical")]
random.seed(42)
random.shuffle(allfiles)
total = len(allfiles)
training = allfiles[0:1116]
validation  = allfiles[1116:1355]
testing = allfiles[1355:]

trainingData = datasetLoader.datasetLoaderClass(training)
validationData = datasetLoader.datasetLoaderClass(validation)
testingData = datasetLoader.datasetLoaderClass(testing)

trainLoader = DataLoader(trainingData, batch_size = 8 , shuffle = True)
validLoader = DataLoader(validationData, batch_size = 8 , shuffle = False)
testLoader = DataLoader(testingData, batch_size = 8 , shuffle = False)



In [ ]:
import torch.optim as optim
import modelArch
import torch.nn as nn

model = modelArch.UNet()
optimizer = optim.Adam(model.parameters(), lr= 3e-4, weight_decay = 1e-3)
scheduler = trc.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, patience=3, factor=0.5
)
weights = [0.0, 0.22750101248873864, 3.8071337283606366, 14.997123180193531, 4.085257960095998]#First Barren wt = 14.99....
criterion = nn.CrossEntropyLoss(weight = trc.tensor(weights, dtype =trc.float32).to(device))

In [52]:
model.to(device)
for epoch in range(20):
    model.train()
    runningLoss = 0.0
    for images, labels in trainLoader:
        optimizer.zero_grad()
        images, labels = images.to(device) , labels.to(device)
        output = model(images)
        loss = criterion(output, labels)
        runningLoss += loss.item()
        loss.backward()
        optimizer.step()
    print(f"Epoch {epoch +1}: Loss: {runningLoss/len(trainLoader):.4f}")
    model.eval()
    valLoss = 0.0
    with trc.no_grad():
        for images, labels in validLoader:
            images = images.to(device)
            labels = labels.to(device)
            output = model(images)
            loss = criterion(output, labels)
            valLoss += loss.item()
    scheduler.step(valLoss)
    print(f"Epoch {epoch+1}: Val Loss: {valLoss/len(validLoader):.4f}")


Epoch 1: Loss: 0.3988
Epoch 1: Val Loss: 0.3667
Epoch 2: Loss: 0.3997
Epoch 2: Val Loss: 0.3758
Epoch 3: Loss: 0.3855
Epoch 3: Val Loss: 0.3674
Epoch 4: Loss: 0.3926
Epoch 4: Val Loss: 0.3605
Epoch 5: Loss: 0.3824
Epoch 5: Val Loss: 0.3924
Epoch 6: Loss: 0.3813
Epoch 6: Val Loss: 0.3638
Epoch 7: Loss: 0.3732
Epoch 7: Val Loss: 0.3528
Epoch 8: Loss: 0.3711
Epoch 8: Val Loss: 0.3571
Epoch 9: Loss: 0.3755
Epoch 9: Val Loss: 0.3617
Epoch 10: Loss: 0.3730
Epoch 10: Val Loss: 0.3633
Epoch 11: Loss: 0.3670
Epoch 11: Val Loss: 0.3756
Epoch 12: Loss: 0.3624
Epoch 12: Val Loss: 0.4194
Epoch 13: Loss: 0.3452
Epoch 13: Val Loss: 0.3561
Epoch 14: Loss: 0.3420
Epoch 14: Val Loss: 0.3497
Epoch 15: Loss: 0.3412
Epoch 15: Val Loss: 0.3525
Epoch 16: Loss: 0.3395
Epoch 16: Val Loss: 0.3685
Epoch 17: Loss: 0.3415
Epoch 17: Val Loss: 0.3674
Epoch 18: Loss: 0.3455
Epoch 18: Val Loss: 0.3584
Epoch 19: Loss: 0.3303
Epoch 19: Val Loss: 0.3539
Epoch 20: Loss: 0.3287
Epoch 20: Val Loss: 0.3554


In [45]:
def calculate_miou(model, loader, num_classes,device):
    model.eval()
    iou_per_class = trc.zeros(num_classes).to(device)
    with trc.no_grad():
        for images, labels in loader:
            images = images.to(device)
            labels = labels.to(device)
            
            output = model(images)
            preds = trc.argmax(output, dim=1)
            for cls in range(num_classes):
                iou=1.0
                predMask = preds==cls
                lblMask = labels==cls
                intersection = (predMask & lblMask).sum()
                union = (predMask | lblMask).sum()
                if union != 0:
                    iou = intersection/union
                else:
                    iou=1
                iou_per_class[cls] += iou
                
    
    return iou_per_class/len(loader), (iou_per_class/len(loader)).mean()

In [59]:
def calculate_accuracy(model, loader, device):
    model.eval()
    correct = 0
    total = 0
    
    with trc.no_grad():
        for images, labels in loader:
            images = images.to(device)
            labels = labels.to(device)
            
            output = model(images)
            preds = trc.argmax(output, dim=1)
            total += labels.size(0)*128*128
            correct += (preds == labels).sum().item()
            # count correct pixels
            # count total pixels
            # hint: (preds == labels).sum()
            # hint: labels.numel() gives total elements
    
    print(correct / total)

In [61]:
checkpoint =trc.load("best_model.pth")
model.load_state_dict(checkpoint['model_state_dict'])
iou_per_class, miou = calculate_miou(model, validLoader, 5, device)
calculate_accuracy(model, testLoader, device)
print(f"Class 0 (Unknown)    : {iou_per_class[0]:.4f}")
print(f"Class 1 (Vegetation) : {iou_per_class[1]:.4f}")
print(f"Class 2 (Urban)      : {iou_per_class[2]:.4f}")
print(f"Class 3 (Barren)     : {iou_per_class[3]:.4f}")
print(f"Class 4 (Water)      : {iou_per_class[4]:.4f}")
print(f"mIoU                 : {miou:.4f}")

0.8657048543294271
Class 0 (Unknown)    : 0.6000
Class 1 (Vegetation) : 0.8628
Class 2 (Urban)      : 0.4279
Class 3 (Barren)     : 0.1797
Class 4 (Water)      : 0.4949
mIoU                 : 0.5131


In [40]:
trc.save({
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'miou': miou,
    'epoch': 40
}, "best_model.pth")
